<a href="https://colab.research.google.com/github/rishithareddym/AI-Lab/blob/main/1BF24CS181_WEEK10_ALBHA_BETA_PRUNING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import math

# Tree structure matching your problem (A to S)
# Added leaf values for terminal nodes K through S
tree = {
    'A': ['B', 'C', 'D'],
    'B': ['E', 'F'],
    'C': ['G', 'H'],
    'D': ['I', 'J'],
    'E': ['K', 'L'],
    'F': ['M'],
    'G': ['N', 'O'],
    'H': ['P'],
    'I': ['Q'],
    'J': ['R', 'S'],
    'K': [4, 5],
    'L': [1, 2],
    'M': [8],
    'N': [9, 3],
    'O': [7, 2],
    'P': [5],
    'Q': [3],
    'R': [1, 0],
    'S': [6]
}

node_values = {}
pruned_edges = set()  # Stores (parent_node, child_index)


def alpha_beta(node, alpha, beta, is_maximizing):
    if isinstance(node, int):
        return node, [str(node)]

    children = tree[node]
    best_path = []

    if is_maximizing:
        best_val = -math.inf
        for i, child in enumerate(children):
            val, path = alpha_beta(child, alpha, beta, False)
            if val > best_val:
                best_val = val
                best_path = [node] + path
            alpha = max(alpha, best_val)
            if alpha >= beta:
                for rem_idx in range(i + 1, len(children)):
                    pruned_edges.add((node, rem_idx))
                break
    else:
        best_val = math.inf
        for i, child in enumerate(children):
            val, path = alpha_beta(child, alpha, beta, True)
            if val < best_val:
                best_val = val
                best_path = [node] + path
            beta = min(beta, best_val)
            if alpha >= beta:
                for rem_idx in range(i + 1, len(children)):
                    pruned_edges.add((node, rem_idx))
                break

    node_values[node] = best_val
    return best_val, best_path


def print_top_down_tree(root):
    coords = {}
    leaf_counter = [0]

    def assign_positions(node, depth=0, is_pruned=False):
        if isinstance(node, int):
            x = leaf_counter[0] * 6 + 4
            leaf_counter[0] += 1
            key = (node, depth, x)
            coords[key] = (x, depth, f"[{node}]", is_pruned)
            return x, key

        children = tree[node]
        child_keys = []
        child_xs = []
        for i, child in enumerate(children):
            c_pruned = is_pruned or ((node, i) in pruned_edges)
            cx, ckey = assign_positions(child, depth + 1, c_pruned)
            child_xs.append(cx)
            child_keys.append((ckey, c_pruned))

        x = (child_xs[0] + child_xs[-1]) // 2
        val_str = str(node_values[node]) if node in node_values else "X"
        label = f"{node}({val_str})"
        coords[node] = (x, depth, label, is_pruned, child_keys)
        return x, node

    assign_positions(root)
    width = leaf_counter[0] * 6 + 8
    max_depth = 4

    roles = ["MAX", "MIN", "MAX", "MIN", "LEAF"]
    for d in range(max_depth + 1):
        node_line = [" "] * width
        branch_line1 = [" "] * width
        branch_line2 = [" "] * width

        for key, info in coords.items():
            x, depth, label = info[0], info[1], info[2]
            if depth == d:
                start = max(0, x - len(label) // 2)
                for idx, ch in enumerate(label):
                    if start + idx < width:
                        node_line[start + idx] = ch

                if len(info) == 5:
                    for ckey, c_pruned in info[4]:
                        cx = coords[ckey][0]
                        mid_x = (x + cx) // 2
                        if cx < x:
                            branch_line1[x - 1] = "/"
                            branch_line2[mid_x] = "X" if c_pruned else "/"
                        elif cx > x:
                            branch_line1[x + 1] = "\\"
                            branch_line2[mid_x] = "X" if c_pruned else "\\"
                        else:
                            branch_line1[x] = "|"
                            branch_line2[x] = "X" if c_pruned else "|"

        print(f"{roles[d]:<5} | " + "".join(node_line).rstrip())
        if d < max_depth:
            print("      | " + "".join(branch_line1).rstrip())
            print("      | " + "".join(branch_line2).rstrip())


# Run Alpha-Beta Search and print the top-down tree
root_val, opt_path = alpha_beta('A', -math.inf, math.inf, True)

print(f"Value of Root Node: {root_val}")
print(f"Optimal Path: {' -> '.join(opt_path)}")
print("Legend: Node(val) | (X) = Unvisited | X on edge = Pruned\n")
print_top_down_tree('A')

Value of Root Node: 4
Optimal Path: A -> B -> E -> K -> 4
Legend: Node(val) | (X) = Unvisited | X on edge = Pruned

MAX   |                                            A(4)
      |                                             / \
      |                                 /              \         \
MIN   |                   B(4)                          C(3)                D(3)
      |                    / \                           / \                 / \
      |                 /       \                     /       X            /     X
MAX   |            E(4)           F(8)           G(3)           H(X)  I(3)         J(X)
      |             / \             |             / \             |     |           / \
      |           /     \           |           /     \           X     |          X   X
MIN   |      K(4)        L(1)     M(8)     N(3)        O(2)     P(X)  Q(3)     R(X)     S(X)
      |       / \         / \       |       / \         / \       |     |       / \       |
      |   